# Лабораторная работа: Генеративные модели
## Часть 1: Character-level токенизация

В этом ноутбуке мы:
1. Скачиваем и предобрабатываем датасет (Shakespeare)
2. Строим char-level словарь
3. Обучаем 4 модели: RNN, LSTM-1слой, LSTM-2слоя, BiLSTM, GPT-like Transformer
4. Сравниваем результаты и генерируем текст

**Датасет**: Tiny Shakespeare (~1MB, 40,000 строк)

**Токенизация**: посимвольная (vocab_size ≈ 65 символов)

In [ ]:
import sys
sys.path.append('..')

import os
import json
import math
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from collections import Counter

from src.data.dataset import download_data, load_text, clean_text, train_val_split, CharTextDataset
from src.tokenizers.char_tokenizer import CharTokenizer
from src.models.rnn_model import SimpleRNN
from src.models.lstm_model import LSTMModel
from src.models.bilstm_model import BiLSTMModel
from src.models.transformer_model import GPTModel
from src.training.trainer import train_model
from src.generation.generate import generate_rnn, generate_transformer, generate_samples
from src.evaluation.metrics import compute_perplexity, bits_per_char
from src.utils.utils import set_seed, get_device, count_parameters, plot_loss_curves, plot_perplexity_curves, plot_comparison

set_seed(42)
device = get_device()
print(f'Устройство: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Загрузка и предобработка данных

In [ ]:
download_data('../data/shakespeare.txt')
raw_text = load_text('../data/shakespeare.txt')

print(f'Размер исходного текста: {len(raw_text):,} символов')
print(f'\nПервые 500 символов:')
print(raw_text[:500])

In [ ]:
text = clean_text(raw_text)

print(f'После очистки: {len(text):,} символов')
print(f'Строк: {text.count(chr(10)):,}')
print(f'Слов: {len(text.split()):,}')

MAX_CHARS = 150_000
text = text[:MAX_CHARS]
print(f'\nИспользуем: {len(text):,} символов')

In [ ]:
char_counts = Counter(text)
most_common = char_counts.most_common(20)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

chars_disp = [repr(c)[1:-1] for c, _ in most_common]
freqs = [f for _, f in most_common]
ax1.bar(range(len(chars_disp)), freqs, color='steelblue')
ax1.set_xticks(range(len(chars_disp)))
ax1.set_xticklabels(chars_disp, rotation=45)
ax1.set_title('Топ-20 символов по частоте')
ax1.set_ylabel('Количество')
ax1.grid(axis='y', alpha=0.3)

letters = sum(v for k, v in char_counts.items() if k.isalpha())
digits  = sum(v for k, v in char_counts.items() if k.isdigit())
spaces  = sum(v for k, v in char_counts.items() if k.isspace())
punct   = sum(v for k, v in char_counts.items() if not k.isalnum() and not k.isspace())

ax2.pie([letters, digits, spaces, punct],
        labels=['Буквы', 'Цифры', 'Пробелы', 'Пунктуация'],
        autopct='%1.1f%%', colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax2.set_title('Состав текста')

plt.tight_layout()
os.makedirs('../outputs/plots', exist_ok=True)
plt.savefig('../outputs/plots/char_distribution.png', dpi=150)
plt.show()

## 2. Character-level токенизация

In [ ]:
tokenizer = CharTokenizer()
tokenizer.build_vocab(text)

print(f'vocab_size: {tokenizer.vocab_size}')
print(f'sample chars: {sorted(tokenizer.char2idx.keys())[:20]}')

os.makedirs('../outputs/metrics', exist_ok=True)
tokenizer.save('../outputs/metrics/char_tokenizer.json')


In [ ]:
sample = 'To be, or not to be, that is the question.'
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)

print(f'Оригинал: {sample}')
print(f'Encoded:  {encoded[:20]}...')
print(f'Decoded:  {decoded}')
print(f'Совпадает: {sample == decoded}')

In [ ]:
train_text, val_text = train_val_split(text, val_fraction=0.1)

SEQ_LEN = 100
BATCH_SIZE = 64

train_ids = tokenizer.encode(train_text)
val_ids   = tokenizer.encode(val_text)

train_dataset = CharTextDataset(train_ids, SEQ_LEN)
val_dataset   = CharTextDataset(val_ids, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train: {len(train_ids):,} символов → {len(train_dataset):,} примеров')
print(f'Val:   {len(val_ids):,} символов → {len(val_dataset):,} примеров')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 3. Обучение моделей

Обучаем 5 моделей на одинаковых данных для честного сравнения:
- **SimpleRNN** — базовая RNN
- **LSTM-1** — однослойный LSTM
- **LSTM-2** — двухслойный LSTM
- **BiLSTM** — двунаправленный LSTM
- **GPT** — Transformer decoder

Гиперпараметры подобраны так, чтобы количество параметров было сопоставимым.

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size
NUM_EPOCHS = 5

models_config = {
    'SimpleRNN': {
        'class': SimpleRNN,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn',
        'lr': 1e-3,
    },
    'LSTM-1layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn',
        'lr': 1e-3,
    },
    'LSTM-2layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn',
        'lr': 1e-3,
    },
    'BiLSTM': {
        'class': BiLSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn',
        'lr': 1e-3,
    },
    'GPT': {
        'class': GPTModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, num_heads=4, num_layers=4, max_seq_len=SEQ_LEN, dropout=0.1),
        'type': 'transformer',
        'lr': 3e-4,
    },
}

print(f"{'Модель':20s} | {'Параметры':>12s}")
print('-' * 36)
for name, cfg in models_config.items():
    m = cfg['class'](**cfg['kwargs'])
    n = count_parameters(m)
    print(f'{name:20s} | {n:>12,}')

In [ ]:
all_histories = {}

for model_name, cfg in models_config.items():
    set_seed(42)
    model = cfg['class'](**cfg['kwargs'])
    
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        model_name=f'char_{model_name}',
        model_type=cfg['type'],
        num_epochs=NUM_EPOCHS,
        lr=cfg['lr'],
        clip_grad=1.0,
        patience=5,
        checkpoint_dir='../outputs/checkpoints',
        device=device,
    )
    all_histories[model_name] = history

print('\nОбучение завершено!')

## 4. Анализ результатов обучения

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs, hist['train_loss'], '--', color=color, alpha=0.7, linewidth=1.5)
    axes[0].plot(epochs, hist['val_loss'], '-', color=color, label=name, linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Кривые обучения (пунктир=train, сплошная=val)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['val_ppl']) + 1)
    axes[1].plot(epochs, hist['val_ppl'], '-o', color=color, label=name, linewidth=2, markersize=4)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/char_training_curves.png', dpi=150)
plt.show()

In [ ]:
rows = []
for name, cfg in models_config.items():
    m = cfg['class'](**cfg['kwargs'])
    hist = all_histories[name]
    best_val_loss = min(hist['val_loss'])
    best_val_ppl = compute_perplexity(best_val_loss)
    avg_time = sum(hist['epoch_times']) / len(hist['epoch_times'])
    
    rows.append({
        'Модель': name,
        'Параметры': f"{count_parameters(m):,}",
        'Best Val Loss': f"{best_val_loss:.4f}",
        'Best Val PPL': f"{best_val_ppl:.1f}",
        'BPC': f"{best_val_loss / math.log(2):.3f}",
        'Сек/эпоха': f"{avg_time:.1f}s",
        'Эпох': len(hist['val_loss']),
    })

df = pd.DataFrame(rows)
print('\nСравнение моделей (char-level токенизация):')
print(df.to_string(index=False))

df.to_csv('../outputs/metrics/char_comparison.csv', index=False)
print('\nМетрики сохранены в outputs/metrics/char_comparison.csv')

In [ ]:
model_names = [r['Модель'] for r in rows]
val_losses = [float(r['Best Val Loss']) for r in rows]
val_ppls = [float(r['Best Val PPL']) for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

bars = axes[0].bar(model_names, val_losses, color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, val_losses):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Итоговый Val Loss')
axes[0].set_ylabel('Loss')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=15)

bars = axes[1].bar(model_names, val_ppls, color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, val_ppls):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9)
axes[1].set_title('Итоговый Val Perplexity')
axes[1].set_ylabel('PPL')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../outputs/plots/char_model_comparison.png', dpi=150)
plt.show()

## 5. Генерация текста

Загружаем лучшие чекпоинты и генерируем текст разными стратегиями.

In [ ]:
def load_best_model(model_name, cfg, device):
    """Загружаем лучший чекпоинт модели."""
    model = cfg['class'](**cfg['kwargs'])
    ckpt_path = f'../outputs/checkpoints/char_{model_name}_best.pt'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'Загружен чекпоинт: {ckpt_path}')
    else:
        print(f'Чекпоинт не найден, используем текущие веса: {ckpt_path}')
    return model.to(device)

prompts = ['To be, or not to be', 'KING HENRY:', 'What light through yonder']
strategies = ['greedy', 'temperature', 'top_k']

all_generations = {}

In [ ]:
for model_name, cfg in models_config.items():
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Модель: {model_name}')
    print('='*60)
    
    model = load_best_model(model_name, cfg, device)
    model_type = cfg['type']
    
    all_generations[model_name] = {}
    
    prompt = 'To be, or not to be'
    for strategy in strategies:
        if model_type == 'transformer':
            gen_text = generate_transformer(model, tokenizer, prompt,
                                            max_new_tokens=150, strategy=strategy, device=device)
        else:
            gen_text = generate_rnn(model, tokenizer, prompt,
                                    max_new_tokens=150, strategy=strategy, device=device)
        
        all_generations[model_name][strategy] = gen_text
        print(f'\n[{strategy}]:')
        print(gen_text[:300])
        print('...')

In [ ]:
os.makedirs('../outputs/generations', exist_ok=True)
with open('../outputs/generations/char_generations.txt', 'w', encoding='utf-8') as f:
    f.write('ГЕНЕРАЦИИ (char-level токенизация)\n')
    f.write('='*60 + '\n\n')
    
    for model_name, gens in all_generations.items():
        f.write(f'\nМодель: {model_name}\n')
        f.write('-'*40 + '\n')
        for strategy, text in gens.items():
            f.write(f'\n[{strategy}]:\n{text}\n')

print('Генерации сохранены в outputs/generations/char_generations.txt')

## 6. Выводы

### Что мы наблюдали:

1. **SimpleRNN** — учится медленнее всего, страдает от vanishing gradient. Генерация часто повторяет короткие паттерны.

2. **LSTM-1layer** — значительно лучше RNN. Умеет удерживать контекст на несколько десятков символов.

3. **LSTM-2layer** — ещё лучше за счёт иерархических представлений. Больше параметров → нужен dropout.

4. **BiLSTM** — хорошо на обучении (видит оба направления), но при генерации работает только forward-направление → менее эффективен для LM.

5. **GPT (Transformer)** — лучший результат при char-level токенизации. Self-attention позволяет явно обращаться к любой позиции в контексте.

### Char-level токенизация:
- **Плюсы**: маленький словарь (65 символов), нет OOV, работает на любом языке
- **Минусы**: длинные последовательности (каждый символ — токен), слабый семантический уровень

### BPC (Bits per Character):
Хорошие char-level языковые модели достигают BPC < 2.0 на Shakespeare.

In [ ]:
print('Лучший результат — GPT Transformer, temperature sampling:')
print('='*60)
if 'GPT' in all_generations:
    print(all_generations['GPT'].get('temperature', 'N/A'))
else:
    print('Модель не обучена в этой сессии')